# QM9 Orientation Bias Demo Tutorial

This notebook walks through the full workflow for the visiualization and quantification of orientation bias:
1. Load QM9 dataset (from torch_geometric) and compute PCA-derived molecular orientations
2. Compare empirical angle distributions to the theoretical SO(3) baseline
3. Visualize results (histogram, reference orientation, Mollweide projection)
4. Estimate KL divergence to the Haar-uniform distribution
5. Train a classifier to detect random rotations
6. Train a regressor on orientation-only features and compare to an uninformative baseline

In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import os
import torch
from lightning import Trainer
from scipy.stats import kstest
from torch_geometric.data import Batch
from torch_geometric.transforms import Compose
from tqdm.auto import tqdm
import warnings
from pyvista.core.errors import PyVistaDeprecationWarning, PyVistaFutureWarning

# PyVista warnings
warnings.filterwarnings("ignore", category=PyVistaDeprecationWarning)
warnings.filterwarnings("ignore", category=PyVistaFutureWarning)

from mol_aligned.data.custom_transforms import (
    RandomJitter,
    ReplaceXWithPCAVectors,
    UniformRandomRotate,
    UniformRandomRotateAngleBounded,
)
from mol_aligned.data.qm9_datamodule import QM9DataModule
from mol_aligned.kl_divergence import estimate_so3_kl
from mol_aligned.mlp_regressor import build_default_regressor
from mol_aligned.mollweide import plot_rotation_matrices_mollweide
from mol_aligned.mpnn_classifier import build_default_classifier
from mol_aligned.orientations import compute_pairwise_angle_distance, compute_pca_batched
from mol_aligned.utils.plot_coordinate_axes import plot_coordinate_axes

In [ ]:
# CONFIGURATION DATAMODULE:
# Path to the preprocessed QM9 data directory used by the datamodule
DATA_DIR = "../data"
# Number of molecules per minibatch during model training/evaluation
BATCH_SIZE = 128
# Number of worker processes for data loading
NUM_WORKERS = 8
# Radius cutoff (in Angstrom) used when building graph edges
RADIAL_CUTOFF_RADIUS = 5.0

## Helper Functions

Small utilities to keep the pipeline cells concise.

In [ ]:
def theoretical_distribution(w: np.ndarray) -> np.ndarray:
    """Return the SO(3) rotation-angle density for Haar-uniform rotations."""
    return 2 / np.pi * np.sin(w / 2) ** 2


def theoretical_cumulative_distribution(w: np.ndarray) -> np.ndarray:
    """Return the cumulative SO(3) rotation-angle distribution."""
    return (w - np.sin(w)) / np.pi


def torch_gaussian_kernel(x: torch.Tensor, mu: float, sigma: float) -> torch.Tensor:
    """Evaluate a normalized Gaussian kernel on a tensor."""
    norm = sigma * np.sqrt(2 * np.pi)
    return torch.exp(-0.5 * ((x - mu) / sigma) ** 2) / norm


def build_qm9_module(
    *,
    split=None,
    custom_transforms=None,
    use_right_split=True,
    selected_features=None,
) -> QM9DataModule:
    """Construct a QM9 datamodule with shared defaults used in this tutorial.

    Args:
        split: Optional train/val/test split ratios.
        custom_transforms: Optional transform pipeline applied per sample.
        use_right_split: Whether to use the repository's canonical split.
        selected_features: Optional list of QM9 target indices.

    Returns:
        Configured QM9DataModule instance.
    """
    # create the data directory if it doesn't exist
    os.makedirs(DATA_DIR, exist_ok=True)

    kwargs = dict(
        data_dir=DATA_DIR,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        self_loops=False,
        radial_cutoff=True,
        radial_cutoff_radius=RADIAL_CUTOFF_RADIUS,
        exclude_edge_attr=False,
    )
    if split is not None:
        kwargs["split"] = split
    if custom_transforms is not None:
        kwargs["custom_transforms"] = custom_transforms
    if use_right_split:
        kwargs["use_right_split"] = True
    if selected_features is not None:
        kwargs["selected_features"] = selected_features
    return QM9DataModule(**kwargs)


def compute_pca_eigenvectors(dataset, n_mols: int, batch_size: int, rotate: bool = False) -> torch.Tensor:
    """Compute PCA orientation matrices for a random subset of molecules.

    Args:
        dataset: QM9 dataset-like object supporting integer indexing.
        n_mols: Number of molecules to sample.
        batch_size: Number of molecules per PCA computation batch.
        rotate: If True, apply random global rotations before PCA.

    Returns:
        Tensor of shape (n_mols, 3, 3) with right-handed PCA axes.
    """
    n_mols = len(dataset) if n_mols is None else n_mols
    random_rotate = UniformRandomRotate()
    perm = np.random.permutation(len(dataset))
    print(f"Loading the molecules...\nThis may take a few minutes (use N_MOLS to control the subset size).")
    molecules = []
    for i in tqdm(perm[:n_mols], desc="Loading molecules"):
        molecules.append(dataset[i])

    pca_eigenvectors = torch.zeros((n_mols, 3, 3))
    num_batches = n_mols // batch_size + (1 if n_mols % batch_size else 0)
    print(f"Processing {n_mols} molecules in {num_batches} batches of size {batch_size}.")

    for batch_idx in tqdm(range(num_batches)):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, n_mols)
        batch_molecules = molecules[start_idx:end_idx]
        if rotate:
            batch_molecules = [random_rotate(mol) for mol in batch_molecules]

        batch = Batch.from_data_list(batch_molecules)
        batch_pca_eigenvectors = compute_pca_batched(
            batch,
            use_mass_weighting=False,
            ensure_right_handed=True,
        )
        pca_eigenvectors[start_idx:end_idx] = batch_pca_eigenvectors

    return pca_eigenvectors


def find_reference_orientation(angles_block: torch.Tensor, gauss_width: float):
    """Select a reference orientation by Gaussian-kernel row scoring.

    The function scores rows in the upper-left quadrant of the angle matrix
    and returns the row index with the largest kernelized density around zero.

    Args:
        angles_block: Pairwise angle-distance matrix.
        gauss_width: Gaussian kernel width used for row scoring.

    Returns:
        Tuple containing selected row index, half-size split index, and
        per-row kernel sums.
    """
    half_size = angles_block.shape[0] // 2
    upper_left = angles_block[:half_size, :half_size]

    dist_kernel_angles = torch_gaussian_kernel(upper_left, 0, gauss_width)
    sum_kernel_angles = dist_kernel_angles.sum(dim=-1)
    argmax_angle = torch.argmax(sum_kernel_angles)

    print(
        f"Argmax row: {argmax_angle}, "
        f"sum of kernel values: {sum_kernel_angles[argmax_angle]}"
    )
    return int(argmax_angle), half_size, sum_kernel_angles

## Step 1: Load Dataset and Compute PCA Orientations

We sample molecules from QM9, compute PCA axes per molecule, then compute pairwise angular distances between orientations.

In [ ]:
# CONFIGURATION
# Number of molecules sampled for the orientation-bias analysis (None to use all QM9 molecules)
N_MOLS = 50000
# Block size used in pairwise angle-distance computation and KL estimation
BLOCK_SIZE = 10000
# Batch size used when computing PCA eigenvectors over sampled molecules
PCA_BATCH_SIZE = 10000
# Whether to treat sign-flipped PCA axes as equivalent orientations
CONSIDER_SIGN_FLIPS = False
# Width (sigma) of Gaussian kernel used to score reference-orientation rows
GAUSS_WIDTH = 0.5

# Use an almost-all-train split to maximize orientation statistics.
qm9_module = build_qm9_module()
qm9_module.setup()
dataset = qm9_module.data_train.dataset
print(f"Length of dataset: {len(dataset)}")

# Compute one orientation matrix (3x3 PCA axes) per sampled molecule.
pca_eigenvectors = compute_pca_eigenvectors(
    dataset=dataset,
    n_mols=N_MOLS,
    batch_size=PCA_BATCH_SIZE,
    rotate=False,
)
print("Done computing PCA eigenvectors.")

# Pairwise angular distances between inferred orientations.
print("Computing pairwise angle-distance matrix...")
angles_block = compute_pairwise_angle_distance(
    pca_eigenvectors,
    block_size=BLOCK_SIZE,
    device="cpu",
    consider_sign_flips=CONSIDER_SIGN_FLIPS,
)
print("Done computing pairwise angle-distance matrix.")

# Pick a representative reference orientation via Gaussian row scoring.
argmax_angle, half_size, sum_kernel_angles = find_reference_orientation(angles_block, GAUSS_WIDTH)
empirical_distances = angles_block[argmax_angle, half_size:]

# Most common orientation:
print(f"Eigenvectors of the most common PC orientation:\n{pca_eigenvectors[argmax_angle]}") 

So, we have found the most common orientation in the QM9 dataset by splitting the data in half and using a kernel density estimate with Gaussian kernels. We split the data in half so that the subsequent Kolmogorov-Smirnov-Test (KS test) is independent from the estimation of the distance distribution relative to the most common reference orientation (see above where we use only the second half of the row: ```empirical_distances = angles_block[argmax_angle, half_size:]```). 

## Step 2: Compare distance distribution between orientations against uniform reference

In [ ]:
# plot the empirical angle distribution relative to the most common orientation:
cmap = mcolors.ListedColormap(["#648fff", "#ffb000", "#dc267f", "#785ef0", "#fe6100", "#000000", "#ffffff"])
x = np.linspace(0, np.pi, 1000)
y = theoretical_distribution(x)

# Compare empirical angles to the theoretical SO(3) angle density.
fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(
    angles_block[argmax_angle].numpy(),
    bins=100,
    alpha=1,
    density=True,
    label="Empirical angle distr.",
    histtype="step",
    color=cmap(0),
)
ax.plot(x, y, label="Theory, random rotations", color="black", linestyle="--")
ax.set_xlabel("Relative rotation angle", fontsize=14)
ax.set_ylabel("Probability", fontsize=14)
ax.legend(loc="upper right", fontsize=12)
ax.tick_params(axis="both", labelsize=12)

# Inset: visualize coordinate axes of the selected reference orientation.
pyvista_arr = plot_coordinate_axes(pca_eigenvectors[argmax_angle].numpy(), res=2000)
pyvista_arr = np.pad(pyvista_arr, ((6, 0), (0, 0), (0, 0)), mode="constant", constant_values=255)
inset_ax = ax.inset_axes([0.03, 0.52, 0.35, 0.5], zorder=-2000)
inset_ax.imshow(pyvista_arr)
inset_ax.axis("off")
ax.text(
    0.04,
    0.5,
    "Reference Orientation",
    transform=ax.transAxes,
    fontsize=11,
    verticalalignment="bottom",
    horizontalalignment="left",
)
plt.title("Empirical vs. Theoretical Angle Distribution", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# formally, run a KS test between empirical distances and the SO(3) theoretical cumulative distribution.
stat, p = kstest(
    empirical_distances,
    cdf=lambda x: theoretical_cumulative_distribution(x),
)
print(f"KS statistic: {stat}, p-value: {p}")

The test statistic is significant (p << 0.05) and therefore we can conclude that the empirical orientations differ significantly from a uniform distribution. Below  

## Step 3: Visualize Orientation Bias
Project all PC eigenvectors of the molecular geometries via an equal area Mollweide projection to visualize the orientation bias.

In [ ]:
# Mollweide projection of all inferred rotation matrices with reference marker.
fig = plot_rotation_matrices_mollweide(
    pca_eigenvectors,
    joint=True,
    marked_axes=pca_eigenvectors[argmax_angle],
    num_bins=1,
    size=4,
    s=2,
    alpha=0.1,
    cmap="viridis",
)
plt.title("2D Mollweide Projection of all PCA Orientations\n", fontsize=14)
plt.show()

We can see a clear structure in the plot of all molecular orientations. By contrast, a uniform distribution of molecules would be preceptually uniform in the Mollweide projection.

## Step 4: Estimate KL Divergence to Uniform SO(3)

Lower values indicate inferred rotations are closer to Haar-uniform.

In [ ]:
kl_div = estimate_so3_kl(angles_block, k=5, block_size=BLOCK_SIZE, verbose=True)
print(f"Estimated KL Divergence to Uniform on SO(3): {kl_div}")

The KL divergence measures the difference to the uniform distribution. The higher the value the larger the difference. 

## Step 5: Train a Rotation-Detection Classifier

The transform pipeline creates lightly perturbed canonical samples and randomly rotated samples, then trains a binary classifier.

In [ ]:
# CONFIGURATION CLASSIFIER TRAINING:
# Toggle expensive model-training sections while exploring geometric analysis
RUN_CLASSIFIER_TRAINING = True

# Training epochs for tutorial runs; increase for stronger final metrics
CLASSIFIER_EPOCHS = 5

if RUN_CLASSIFIER_TRAINING:
    # Compose perturbations so the model learns to separate canonical vs rotated samples.
    custom_transforms_cls = Compose([
        RandomJitter(p=0.0, sigma_max=0.5, sigma_min=0., clip=None),  # switch to p=1.0 for random jitter
        UniformRandomRotateAngleBounded(
            add_is_transformed_flag=False,
            p=0.0, # switch to p=1 to preturb the canonical orientations with small rotations
            max_angle=10.0, 
            min_angle=0.0,
        ),
        UniformRandomRotate(add_is_transformed_flag=True, p=0.5),
    ])

    qm9_cls = build_qm9_module(
        custom_transforms=custom_transforms_cls,
        use_right_split=True,
    )
    qm9_cls.setup()

    model_cls = build_default_classifier(
        loss_function=torch.nn.BCEWithLogitsLoss(),
        max_epochs=CLASSIFIER_EPOCHS,
    )

    # Short tutorial training run (increase CLASSIFIER_EPOCHS for real experiments).
    trainer_cls = Trainer(max_epochs=CLASSIFIER_EPOCHS, min_epochs=CLASSIFIER_EPOCHS, gradient_clip_val=0.5)
    trainer_cls.fit(model_cls, qm9_cls)
    print("Train metrics:", trainer_cls.callback_metrics)

    trainer_cls.test(model_cls, qm9_cls)
    print("Test metrics:", trainer_cls.callback_metrics)
else:
    print("Skipping classifier training (RUN_CLASSIFIER_TRAINING=False).")

Using only a tiny message passing neural network (238 parameters!), already after a few epochs the classification accuracy is clearly > 50%. This shows that a neural classifier can easily distinguish samples in canonical orientation from randomly rotated ones.

## Step 6: Orientation-Only Regressor

Here we replace node features with PCA vectors to test how much predictive signal orientation bias alone provides.

In [ ]:
# CONFIGURATION REGRESSOR TRAINING:
# Toggle expensive model-training sections while exploring geometric analysis
RUN_REGRESSOR_TRAINING = True
# Training epochs for tutorial runs; increase for stronger final metrics
REGRESSOR_EPOCHS = 5
# QM9 target indices to predict; [6] corresponds to ZPVE
SELECTED_FEATURES = [6]

if RUN_REGRESSOR_TRAINING:
    # Replace node features with PCA orientation vectors to isolate orientation signal.
    custom_transforms_reg = Compose([
        ReplaceXWithPCAVectors(use_mass_weighting=False),
    ])

    qm9_reg = build_qm9_module(
        custom_transforms=custom_transforms_reg,
        use_right_split=True,
        selected_features=SELECTED_FEATURES,
    )
    qm9_reg.setup()

    model_reg = build_default_regressor(
        loss_function=torch.nn.MSELoss(),
        max_epochs=REGRESSOR_EPOCHS,
    )

    trainer_reg = Trainer(max_epochs=REGRESSOR_EPOCHS, min_epochs=REGRESSOR_EPOCHS, gradient_clip_val=0.5)
    trainer_reg.fit(model_reg, qm9_reg)
    print("Train metrics:", trainer_reg.callback_metrics)

    trainer_reg.test(model_reg, qm9_reg)
    test_metrics = trainer_reg.callback_metrics
    print("Test metrics:", test_metrics)

    # Baseline: always predict the test-set mean target.
    test_targets = np.concatenate([data.y.numpy() for data in qm9_reg.data_test], axis=0)
    mean_target = test_targets.mean(axis=0)
    mse_uninformative = np.mean((test_targets - mean_target) ** 2)
    print(f"Best MSE for uninformative model: {mse_uninformative}")

    mse_trained = test_metrics["test/loss"].item()
    relative_improvement = (mse_trained -  mse_uninformative) / mse_uninformative
    print(
        "Relative improvement over uninformative baseline (percent): "
        f"{relative_improvement * 100:.2f}%"
    )
else:
    print("Skipping regressor training (RUN_REGRESSOR_TRAINING=False).")

The MLP that receives only the principal components as input is able to learn a patter to predict the Zero Point Vibrational Energy. 
For a completely uniformative input (in the absence of orientation bias), the best the model could do is predict the mean of the data.
This simple experiment shows that neural networks can in principle exploit orientation bias to artificially inflate their performance. 